### ЗАДАЧА: Резервирование товаров по складам

Команда закупок получает пакет заявок на резерв товаров по складам.
Нужно собрать систему, которая:
- принимает корректные заявки,
- отбрасывает неправильные или невыполнимые запросы,
- обновляет остатки на складе после успешного резерва,
- собирает отдельный журнал ошибок,
- позволяет быстро понять, кто зарезервировал больше всего товара и на каком складе осталось меньше всего позиций.

В части строк указаны неизвестные склады или товары,
в части количество заполнено неверно,
а некоторые заявки пытаются зарезервировать больше доступного остатка или дублируют уже обработанный `request_id`.


In [4]:
from dataclasses import dataclass
from typing import Optional
import math

stocks = {
    'MSK-1': {'keyboard': 10, 'mouse': 20, 'monitor': 4},
    'SPB-2': {'keyboard': 6, 'dock': 5, 'monitor': 2},
    'KZN-3': {'mouse': 7, 'dock': 3, 'laptop': 2},
}

# rows: request_id|client|warehouse_id|sku|quantity
rows = [
    'RQ-100|Acme|MSK-1|keyboard|3',
    'RQ-101|Beta|SPB-2|dock|2',
    'RQ-102|Acme|MSK-1|monitor|5',
    'RQ-103|Delta|X-999|mouse|1',
    'RQ-104|Gamma|KZN-3|laptop|0',
    'RQ-105|Beta|SPB-2|chair|1',
    'RQ-101|Beta|MSK-1|mouse|4',
    'RQ-106|Acme|MSK-1|mouse|7',
    'RQ-107|Kira|KZN-3|laptop|1',
]


class ReservationError(Exception):
    pass


class RowFormatError(ReservationError):
    pass


class WarehouseNotFoundError(ReservationError):
    pass


class ProductNotFoundError(ReservationError):
    pass


class QuantityError(ReservationError):
    pass


class StockLimitError(ReservationError):
    pass


class DuplicateRequestError(ReservationError):
    pass


@dataclass(order=True)
class ReservationRequest:
    request_id: str
    client: str
    warehouse_id: str
    sku: str
    quantity: int


class Warehouse:
    def __init__(self, warehouse_id, products):
        # TODO: сохранить warehouse_id
        # TODO: создать отдельную копию словаря products
        # TODO: создать список reservations
        self.warehouse_id = warehouse_id
        self.products = products
        self.reservations = []

    def has_sku(self, sku):
        # TODO: вернуть True/False, есть ли такой sku в self.products
        return sku in self.products.keys()

    def available(self, sku):
        # TODO: вернуть текущий остаток по sku
        return self.products[sku]

        

    def reserve(self, request):
        # TODO: если request.sku отсутствует -> raise ProductNotFoundError(...)
        # TODO: если request.quantity > available(...) -> raise StockLimitError(...)
        # TODO: уменьшить остаток на складе
        # TODO: добавить request в self.reservations
        if request.sku not in self.products:
            raise ProductNotFoundError('такого продукта не существует!')
        if request.quantity > self.available(request.sku):
            raise StockLimitError('превышен лимит товара!')
        self.products[request.sku] -= request.quantity
        self.reservations.append(request)

    def total_left(self):
        # TODO: вернуть сумму всех остатков на складе
        m = 0
        for el in self.products.values():
            m += el
        return m


    def reserved_total(self):
        # TODO: вернуть сумму quantity по всем self.reservations
        c = 0
        for el in self.reservations:
            c += el.quantity
        return c



class ReservationService:
    def __init__(self, stocks):
        # TODO: создать warehouses вида warehouse_id -> Warehouse(...)
        # TODO: создать списки accepted и errors
        # TODO: создать множество processed_ids
        self.warehouses = {}
        for key, val in stocks.items():
            self.warehouses[key] = Warehouse(warehouse_id=key, products=val)
            
            
        self.accepted = []
        self.errors = []
        self.processed_ids = set()

    def parse_request(self, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: request_id, client, warehouse_id, sku, quantity_raw
        # TODO: quantity_raw преобразовать в int
        # TODO: если warehouse_id не существует -> WarehouseNotFoundError
        # TODO: если quantity <= 0 -> QuantityError
        # TODO: вернуть объект ReservationRequest(...)
        arr = row.split('|')
        request_id, client, warehouse_id, sku, quantity_raw = arr[0], arr[1], arr[2], arr[3], int(arr[4])
        if warehouse_id not in self.warehouses.keys():
            raise WarehouseNotFoundError('такой склад не существует')
        if quantity_raw <= 0:
            raise QuantityError('количество должно быть больше нуля!')
        return ReservationRequest(request_id=request_id, client=client, warehouse_id=warehouse_id, sku=sku, quantity=quantity_raw)
        

    def submit(self, row):
        # TODO: внутри try вызвать parse_request(row)
        # TODO: если request.request_id уже в processed_ids -> DuplicateRequestError
        # TODO: затем warehouses[request.warehouse_id].reserve(request)
        # TODO: после успеха добавить request_id в processed_ids
        # TODO: добавить request в self.accepted
        # TODO: ReservationError сохранить в self.errors как (row, error_type, message)
        try:
            req = self.parse_request(row)
            if req.request_id in self.processed_ids:
                raise DuplicateRequestError('request_id уже обработано!')
            self.warehouses[req.warehouse_id].reserve(req)
            self.processed_ids.add(req.request_id)
            self.accepted.append(req)
        except ReservationError as e:
            self.errors.append((row, type(e).__name__, e))
            


    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for el in rows:
            self.submit(el)

    def client_totals(self):
        # TODO: собрать dict вида client -> total_reserved_quantity
        obj = {}
        for el in self.accepted:
            obj.setdefault(el.client, 0)
            obj[el.client] += el.quantity
        return obj

    def top_client(self):
        # TODO: использовать client_totals()
        # TODO: вернуть tuple(client, total_quantity) с максимумом
        res = self.client_totals()
        m = 0
        cl = None
        for key, val in res.items():
            if val > m:
                m = val
                cl = key
        return (cl, m)


    def lowest_stock_warehouse(self):
        # TODO: найти склад с минимумом total_left()
        # TODO: вернуть tuple(warehouse_id, total_left)
        m = math.inf
        wareh = None
        for wrh, val in self.warehouses.items():
            if val.total_left() < m:
                m = val.total_left()
                wareh = wrh
        return (wareh, m)
            


    def warehouse_snapshot(self):
        # TODO: собрать dict вида warehouse_id -> копия текущих остатков products
        res = {}
        for key, val in self.warehouses.items():
            res[key] = val.products
        return res

    def find_request(self, request_id) -> Optional[ReservationRequest]:
        # TODO: вернуть Optional[ReservationRequest]
        # TODO: пройтись по self.accepted и найти нужную заявку
        # TODO: если не найдено -> вернуть None
        for el in self.accepted:
            if el.request_id == request_id:
                return el
        return None

        


service = ReservationService(stocks)

# TODO: загрузить rows через service.load(rows)
# TODO: вывести принятые заявки
# TODO: вывести ошибки
# TODO: вывести warehouse_snapshot()
# TODO: вывести client_totals()
# TODO: вывести top_client()
# TODO: вывести lowest_stock_warehouse()
# TODO: вывести find_request('RQ-107')

service.load(rows)

print(f'Принятые заявки: {service.accepted}')
print(f'--------------------------------')
print(f'Ошибки: {service.errors}')
print('-----------')
print(service.warehouse_snapshot())
print('-----------')
print(service.client_totals())
print('-----------')
print({service.top_client()})
print('-----------')
print(f'склад с минимумом остатка: {service.lowest_stock_warehouse()}')
print('-----------')
print(f'Заявка RQ-107: {service.find_request('RQ-107')}')




Принятые заявки: [ReservationRequest(request_id='RQ-100', client='Acme', warehouse_id='MSK-1', sku='keyboard', quantity=3), ReservationRequest(request_id='RQ-101', client='Beta', warehouse_id='SPB-2', sku='dock', quantity=2), ReservationRequest(request_id='RQ-106', client='Acme', warehouse_id='MSK-1', sku='mouse', quantity=7), ReservationRequest(request_id='RQ-107', client='Kira', warehouse_id='KZN-3', sku='laptop', quantity=1)]
--------------------------------
Ошибки: [('RQ-102|Acme|MSK-1|monitor|5', 'StockLimitError', StockLimitError('превышен лимит товара!')), ('RQ-103|Delta|X-999|mouse|1', 'WarehouseNotFoundError', WarehouseNotFoundError('такой склад не существует')), ('RQ-104|Gamma|KZN-3|laptop|0', 'QuantityError', QuantityError('количество должно быть больше нуля!')), ('RQ-105|Beta|SPB-2|chair|1', 'ProductNotFoundError', ProductNotFoundError('такого продукта не существует!')), ('RQ-101|Beta|MSK-1|mouse|4', 'DuplicateRequestError', DuplicateRequestError('request_id уже обработано!